## 1. Data Wrangling
Explanation of what this section does (load data, compute firing rates, etc.)

In [ ]:
import numpy as np
import pandas as pd

# Load datasets (adjust paths if needed)
spikes = pd.read_csv("../data/spikes.csv")
trials = pd.read_csv("../data/trials.csv")
neurons = pd.read_csv("../data/neurons.csv")

# Inspect the data
print(spikes.head())
print(trials.head())
print(neurons.head())

# Time window for firing rate calculation
window_start = 0
window_end = 0.5

# Function to compute firing rate for a neuron in a trial
def compute_firing_rate(spike_times, start, end):
    spikes_in_window = spike_times[(spike_times >= start) & (spike_times <= end)]
    return len(spikes_in_window) / (end - start)

# Build population activity matrix
firing_rates = []

for trial in trials.itertuples():
    trial_rates = []
    for neuron_id in neurons['neuron_id']:
        neuron_spikes = spikes[spikes['neuron_id'] == neuron_id]['spike_time'].values
        rate = compute_firing_rate(neuron_spikes, trial.stim_onset + window_start, trial.stim_onset + window_end)
        trial_rates.append(rate)
    firing_rates.append(trial_rates)

# Convert to NumPy array
X = np.array(firing_rates)
y = trials['choice'].values

print("Population matrix shape:", X.shape)


## 2. Integrating Neural Activity with Anatomical Regions
Explanation of region integration

In [ ]:
region_labels = neurons['region'].values

# Convert population activity to DataFrame
activity_df = pd.DataFrame(X)
activity_df.columns = neurons['neuron_id']
activity_df['choice'] = y

print(activity_df.head())

## 3. Dimensionality Reduction (PCA)
Explain PCA and why you’re using it

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='coolwarm', alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Neural Population Activity PCA")
plt.colorbar(label='Behavioral Choice')
plt.show()

## 4. Decoding Behavioral Choice
Explain logistic regression decoding

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='coolwarm', alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Neural Population Activity PCA")
plt.colorbar(label='Behavioral Choice')
plt.show()

## 5. Comparing Brain Regions
Explain why you’re decoding per region

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000)

accuracy = cross_val_score(model, X, y, cv=5, scoring='accuracy')
auc = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

print("Mean accuracy:", accuracy.mean())
print("Mean AUC:", auc.mean())

## 6. Permutation Testing
Explain shuffling labels to get chance-level performance

In [ ]:
region_performance = {}

for region in neurons['region'].unique():
    region_neurons = neurons[neurons['region'] == region]['neuron_id']
    region_X = activity_df[region_neurons].values
    
    if region_X.shape[1] > 2:  # need at least 3 neurons to train
        scores = cross_val_score(model, region_X, y, cv=5)
        region_performance[region] = scores.mean()

print("Region-specific decoding accuracy:")
for region, acc in region_performance.items():
    print(f"{region}: {acc:.3f}")